In [1]:
!mkdir going_modular  #creating a folder named "going_modular"

In [2]:
import os
import requests
import zipfile
from pathlib import Path

#setup path to datafolder

data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

#if image folder doesnt exits, download it and prepare it...
if image_path.is_dir():
  print(f"The {image_path} directory already exists")
else:
  print(f"The {image_path} directory doesnt exist, creating one...")
  image_path.mkdir(parents=True, exist_ok=True)

#download pizza steak sushi data
with open (data_path /"pizza_steak_sushi.zip", "wb") as f:
  requests = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
  print(f"Downloading pizza, steak , sushi data....")
  f.write(requests.content)

#unzip pizza , steak and sushi data
with zipfile.ZipFile(data_path/"pizza_steak_sushi.zip","r") as zip_ref:
  print(f"Unzipping the pizza , steak and sushi data")
  zip_ref.extractall(image_path)

#remove zip file
os.remove(data_path/"pizza_steak_sushi.zip")

The data/pizza_steak_sushi directory doesnt exist, creating one...
Unzipping the pizza , steak and sushi data


In [3]:
%%writefile going_modular/data_setup.py
#creates a file named "data_setup.py" inside the folder going_modular

""" contains functionality for creating  PyTorch Dataloaders
for image classification data """

import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir : str,
    test_dir : str,
    transform : transforms.Compose,
    batch_size: int,
    num_workers: int=NUM_WORKERS
):

  #use ImageFolder to create datasets
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform =transform)

  #get class namaes
  class_names = train_data.classes

  #turn image into dataloaders

  train_dataloader = DataLoader(train_data,
                              batch_size=batch_size,
                              shuffle=True,
                              num_workers=num_workers,
                              pin_memory=True)

  test_dataloader = DataLoader(test_data,
                              batch_size=batch_size,
                              shuffle=False, #dont need to shuffle test dataset
                              num_workers=num_workers,
                              pin_memory=True)

  return train_dataloader , test_dataloader , class_names




Writing going_modular/data_setup.py


In [4]:
%%writefile going_modular/model_builder.py

"""Contains pyTorch model code to instantiate the TinyVGG model """

import torch
from torch import nn

class TinyVGG_model(nn.Module):
  """ Creates TinyVGG architecture

  Replicates the TinyVGG architecture from the CNN explainer website in PyTorch.
  See the original architecture here: https://poloclub.github.io/cnn-explainer/

  Args:
  input_shape : int indicating number of input channels
  hidden_units = int indicating number of hidden units between layers
  output layers : int indicating number of output units  """

  def __init__(self, input_shape:int, hidden_units:int , output_shape: int) -> None:
    super().__init__()
    self.conv_block1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)

    )
    self.conv_block2 = nn.Sequential(
        nn.conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
        nn.ReLU(),
        nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    nn.classifer(nn.Sequential(
        nn.Flatten(),
        #where did this in_features shape come from?
        #its because each layer of our network compresses and change the shape of our input data
        nn.Linear(in_features=hidden_units*16*16)
    ))


  def forward(self, x:torch.Tensor)-> torch.Tensor:
    return self.classifer(self.conv_block2(self.conv_block1(x)))  #leveraging the benefit of operation fusion






Writing going_modular/model_builder.py


In [5]:
 import torch
#import model_builder.py
from going_modular import model_builder  #we are importing the files we created inside the going_modular folder

device = "cuda" if torch.cuda.is_available() else "cpu"




In [6]:
%%writefile going_modular/engine.py

import torch
from tqdm.auto import tqdm
from typing import Dict, List, Tuple


def train_step(model:torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module,
               optimizer:torch.optim.Optimizer,
               device: torch.device)-> tuple[float, float]:
  #put the model in correct device
  model.to(device)



  train_loss , train_accuracy = 0,0

  for batch, (x,y) in enumerate(dataloader):
    #send the data to target device
    x , y = x.to(device) , y.to(device)

    #1. Forward pass
    y_logits = model(x)

    #calculate loss
    loss = loss_fn(y_logits, y)
    tran_loss += loss.item()

    #optimizer zero grad
    optimizer.zero_grad()

    #loss backward
    loss.backward()

    #opotimizer step
    optimizer.step()

    #calculate and accumulate the accuracy metrics throughout all the batches
    pred_prob_Class = torch.argmax(torch.softmax(y_logits, dim=1), dim=1)
    train_accuracy += ((pred_prob_Class == y).sum).item() / len(y_logits)

  #Adjusted train loss and accuracy throughout the train process
  train_loss /= len(dataloader)
  train_accuracy /= len(dataloader)

  return train_loss , train_accuracy

def test_step(model:torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn : torch.nn.Module,
              optimizer : torch.optim.Optimizer,
              device: torch.device
              ) -> tuple[float , float]:


  #out the model in evaluation mode
  model.eval()

  #set the test loss and test accuracy metrics

  test_loss , test_accuracy = 0,0

  #Turn on inference context manager
  with torch.inference_mode():
    for batch , (x,y) in enumerate(dataloader):

      #send  data to the target device
      x , y = x.to(device) , y.to(device)

      #forward pass
      y_pred = model(x)

      #calculate loss
      loss = loss_fn(y_pred, y)
      test_loss += loss.item()

      #calculate the pred-logits into pred probability
      pred_prob = torch.softmax(y_pred, dim=1)

      #calculate the pred class from the pred probability
      pred_prob_class = torch.argmax(y_pred, dim=1)
      test_accuracy += ((pred_prob_class == y).sum).item() / len(y_pred)


  #Adjust the average loss and accuracy per batch in the whole dataset
  test_loss /= len(dataloader)
  test_accuracy /= len(dataloader)

  return test_loss , test_accuracy


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          epochs: int,
          device: torch.device) -> dict[str, int]:


          #creates an empty resulys dictionary
          results = {"train_loss":[],
                     "train_accuracy":[],
                     "test_loss":[],
                     "test_accuracy":[]}

          #loop through training and testing for number of epochs
          for epoch in tqdm(range(epochs)):
            train_loss , train_accuracy = train_step(model=model,
                                                dataloader=train_dataloader,
                                                loss_fn=loss_fn,
                                                optimizer=optimizer,
                                                device=device)

            test_loss , test_accuracy = test_step(model=model,
                                                  dataloader=test_dataloader,
                                                  loss_fn=loss_fn,
                                                  optimizer=optimizer,
                                                  device=device
                                                  )

            #print out whats happening
            print(f"Epoch:{epoch+1}"|
                  f"train_loss:{train_loss}"|
                  f"train_accuracy:{train_accuracy}"|
                  f"test_loss:{test_loss}"|
                  f"test_accuracy:{test_accuracy}")

            #Now update the results dictionary

            results["train_loss"].append(train_loss)
            results["train_accuracy"].append(train_accuracy)
            results["test_loss"].append(test_loss)
            results["test_accuracy"].append(test_accuracy)

          #return the filled results at the end of the epoch
          return results






Writing going_modular/engine.py


In [7]:
#now we can import the engine.py from going_modular folder

from going_modular import engine

In [8]:
%%writefile going_modular/utils.py

import torch
from pathlib import Path
def save_model(model: torch.nn.Module,
               targ_dir: str,
               model_name: str):
  """ saves the model of model_name in target directory"""

  #create target directory
  target_dir_path = Path(targ_dir)
  target_dir_path.mkdir(parents=True,
                        exist_ok=True)

  #create model save path
  assert model_name.endswith(".pth") or model_name.endswith(".pt") , "model name should end with .pth or .pt"
  model_save_path = target_dir_path / model_name

  #save the model.state_dict()

  print(f"[INFo] saving the model into model_save_path")
  torch.save(obj=model.state_dict(),
             f=model_save_path())

Writing going_modular/utils.py


In [9]:
#now we can use this file and function inside this file to save the model

from going_modular import utils

In [10]:
%%writefile going_modular/train.py

import os
import torch
import data_setup, engine , model_builder , utils

from torchvisions import transformers

#setup the hyperparameters
NUM_EPOCHS = 5
BATCH_SIZE= 32
HIDDEN_UNITS=10
LEARNING_RATE = 0.001

#setup directories
train_dir = "data/pizza_steak_sushi/train"
test_dir = "data/pizza_steak_sushi/test"

#setup target device
device = "cuda" if torch.cuda.is_available() else "cpu"

#create transforms
data_transform = transforms.Compose([transforms.Resize(64,64),
                                     trasforms.ToTensor()])

#Create Dataloaders with the help from data_setup.py
train_dataloader , test_dataloader , class_names = data_setup.create_dataloaders(
                                                    train_dir = train_dir,
                                                    test_dir = test_sir,
                                                    transform = data_transform,
                                                    batch_size = BATCH_SIZE
                                                                                 )

#create model with help from model_builder.py

model = model_builder.TinyVGG_model(
                      input_shape=3,
                      hidden_units = HIDDEN_UNITS,
                      output_shape = len(class_names)).to(device)

#set loss and optimizer
loss_fn = nn.torch.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameter,
                             lr=LEARNING_RATE)

#start training with help from engine.poy
engine.train(model=model,
             train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             epochs=NUM_EPOCHS,
             device=device)


#save the model with help from utils.py
utils.save_model(model=model,
                 targ_dir="models",
                 model_name="model_1.pth")






Writing going_modular/train.py
